<a href="https://colab.research.google.com/github/miruts-code/examples/blob/main/pytorch_language_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Compute perplexity on the test set (Transformer version)

In [ ]:
import torch.nn.functional as F
import math

def batchify(data_source, bsz, device):
    nbatch = data_source.size(0) // bsz
    data_source = data_source.narrow(0, 0, nbatch * bsz)
    data_source = data_source.view(bsz, -1).t().contiguous()
    return data_source.to(device)

def get_batch(source, i, bptt=35):
    seq_len = min(bptt, len(source) - 1 - i)
    data = source[i:i+seq_len]
    target = source[i+1:i+1+seq_len].reshape(-1)
    return data, target

def evaluate_perplexity_transformer(eval_model, data_source, ntokens, device, bptt=35):
    eval_model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for i in range(0, data_source.size(0) - 1, bptt):
            data, targets = get_batch(data_source, i, bptt)
            output = eval_model(data)
            output_flat = output.view(-1, ntokens)
            total_loss += len(data) * F.nll_loss(F.log_softmax(output_flat, dim=1), targets).item()
    avg_loss = total_loss / (len(data_source) - 1)
    return avg_loss, math.exp(avg_loss)

eval_batch_size = 10
test_data = batchify(corpus.test, eval_batch_size, device)

avg_loss, ppl = evaluate_perplexity_transformer(base_model, test_data, ntokens, device)
print(f"Base model — Loss: {avg_loss:.4f}, Perplexity: {ppl:.3f}")

Base model — Loss: 7.1384, Perplexity: 1259.376


Compute parameter count and model size, then load the trained Transformer back in

In [ ]:
import torch
import data
import os

corpus = data.Corpus('./data/wikitext-2')
ntokens = len(corpus.dictionary)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with open('model.pt', 'rb') as f:
    base_model = torch.load(f, map_location=device, weights_only=False)

base_model.eval()

param_count = sum(p.numel() for p in base_model.parameters())
print(f"Base model param count: {param_count:,}")

model_size_mb = os.path.getsize('model.pt') / (1024 ** 2)
print(f"Base model size on disk: {model_size_mb:.2f} MB")

Base model param count: 17,863,678
Base model size on disk: 73.05 MB


Train the baseline transformer model



In [ ]:
!python main.py --accel --model Transformer --emsize 256 --nhid 256 --nlayers 2 --nhead 4 --epochs 20 --save model.pt

Using device: cuda
/usr/local/lib/python3.13/dist-packages/torch/nn/modules/transformer.py:144: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  self.encoder = TransformerEncoder(
| epoch   1 |   200/ 2983 batches | lr 20.00 | ms/batch 20.40 | loss 18.19 | ppl 79562457.74
| epoch   1 |   400/ 2983 batches | lr 20.00 | ms/batch 16.15 | loss 15.60 | ppl 5968223.43
| epoch   1 |   600/ 2983 batches | lr 20.00 | ms/batch 16.25 | loss 11.15 | ppl 69347.68
| epoch   1 |   800/ 2983 batches | lr 20.00 | ms/batch 17.03 | loss 10.50 | ppl 36369.36
| epoch   1 |  1000/ 2983 batches | lr 20.00 | ms/batch 16.30 | loss  9.94 | ppl 20821.22
| epoch   1 |  1200/ 2983 batches | lr 20.00 | ms/batch 16.33 | loss  9.36 | ppl 11613.41
| epoch   1 |  1400/ 2983 batches | lr 20.00 | ms/batch 16.53 | loss  8.92 | ppl  7491.14
| epoch   1 |  1600/ 2983 batches | lr 20.00 |

Inspect the data and model files

In [6]:
!head -n 30 model.py
!head -n 30 data.py

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class RNNModel(nn.Module):
    """Container module with an encoder, a recurrent module, and a decoder."""

    def __init__(self, rnn_type, ntoken, ninp, nhid, nlayers, dropout=0.5, tie_weights=False):
        super(RNNModel, self).__init__()
        self.ntoken = ntoken
        self.drop = nn.Dropout(dropout)
        self.encoder = nn.Embedding(ntoken, ninp)
        if rnn_type in ['LSTM', 'GRU']:
            self.rnn = getattr(nn, rnn_type)(ninp, nhid, nlayers, dropout=dropout)
        else:
            try:
                nonlinearity = {'RNN_TANH': 'tanh', 'RNN_RELU': 'relu'}[rnn_type]
            except KeyError as e:
                raise ValueError( """An invalid option for `--model` was supplied,
                                 options are ['LSTM', 'GRU', 'RNN_TANH' or 'RNN_RELU']""") from e
            self.rnn = nn.RNN(ninp, nhid, nlayers, nonlinearity=nonlinearity, dropout=dropout)
        self

creating results folder in the repo. and install dependencies

In [5]:
import os
import torch
RESULTS_DIR = "results"
os.makedirs(RESULTS_DIR, exist_ok=True)
print(os.listdir("."))
print(torch.__version__)

['data.py', 'README.md', 'main.py', 'data', 'generate.py', 'requirements.txt', 'results', 'model.py']
2.11.0+cu128


confirm gpu is available

In [3]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device name: {torch.cuda.get_device_name(0)}")

CUDA available: True
Device name: Tesla T4


cloing repo and set working diretory

In [2]:
!git clone https://github.com/miruts-code/examples.git
%cd examples/word_language_model
!ls

Cloning into 'examples'...
remote: Enumerating objects: 4274, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 4274 (delta 8), reused 4 (delta 4), pack-reused 4256 (from 2)
Receiving objects: 100% (4274/4274), 17.89 MiB | 15.78 MiB/s, done.
Resolving deltas: 100% (2146/2146), done.
/content/examples/word_language_model
data  data.py  generate.py  main.py  model.py  README.md  requirements.txt
